<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/airbnb_miniproject_AA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Airbnb Mini-Project: Price Prediction with Neural Networks
**SCH-MGMT 661: Applications of AI Models**

This project extends Assignment 1 by comparing different neural network architectures and generating business-relevant insights for Airbnb price prediction.

---
## 1. Data Preparation
Reusing the cleaned dataset from Assignment 1 with additional features.

### 1.1 Data Loading and Initial Exploration

In [ ]:
# Import required libraries for data analysis and visualization

import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns            # for enhanced visualizations

# Set a default aesthetic style for plots
sns.set(style="whitegrid")

In [ ]:
# import Airbnb Listing datasets for Asheville

listings_url = 'https://data.insideairbnb.com/united-states/nc/asheville/2024-06-21/data/listings.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip')

Loading the dataset for data exploration and cleaning. Taken directly from Assignment 1.

In [ ]:
# Display the first 10 rows
listings_df.head(10)

In [ ]:
# Explore columns, data types, and non-null counts
listings_df.info()

In [ ]:
# Display the shape of the DataFrame (rows, columns)
listings_df.shape

Initial data exploration. Taken directly from Assignment 1.

### 1.2 Filter Listings for Asheville

In [ ]:
# Print all unique host locations in the dataset
print(listings_df['host_location'].unique())      # List all location names
print(len(listings_df['host_location'].unique())) # Total number of unique locations

In [ ]:
# Filter Listings for Asheville
asheville_df = listings_df[listings_df['host_location'] == 'Asheville, NC']

In [ ]:
# Shape of the filtered dataset
asheville_df.shape

Ensure listings belong to hosts located in Asheville, NC. Kept all records with host_location as Asheville, dropped all rows with missing or different host_location values. Taken directly from Assignment 1.

### 1.3 Select Relevant Features

In [ ]:
# Define the columns we want to keep and create a new dataframe with the selected columns
selected_columns = [
    'price', 'bathrooms','bedrooms', 'number_of_reviews',
    'room_type', 'host_identity_verified', 'host_is_superhost',
    'accommodates', 'review_scores_rating'
]

asheville_df = asheville_df[selected_columns]

In [ ]:
# Preview the first few rows of the new dataframe
asheville_df.head()

Extended the selected features from W3 to include accommodates, review_scores_rating, and room_type. Taken directly from Assignment 1.

### 1.4 Handle Missing Values

In [ ]:
# Check for missing values in each column
asheville_df.isnull().sum()

Check for missing values before applying imputation strategies. Taken directly from Assignment 1.

In [ ]:
# Drop rows where the target variable (price) is missing and then re-check for missing values
asheville_df = asheville_df.dropna(subset=['price'])
asheville_df.isnull().sum()

Drop rows where the price value is missing. Taken directly from Assignment 1.

In [ ]:
# Impute host_is_superhost using mode
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].fillna(
    asheville_df['host_is_superhost'].mode()[0]
)
asheville_df.isnull().sum()

Impute host_is_superhost using mode. Taken directly from Assignment 1.

In [ ]:
# Impute numerical missing values using mean or median, depending on data distribution
asheville_df['bedrooms'] = asheville_df['bedrooms'].fillna(asheville_df['bedrooms'].median())
asheville_df['bathrooms'] = asheville_df['bathrooms'].fillna(asheville_df['bathrooms'].mean())
asheville_df.isnull().sum()

Impute bedrooms using median and bathrooms using mean to fill missing values for numerical features except for review_scores_rating. Taken directly from Assignment 1.

In [ ]:
# For review_scores_rating, replace missing values with 0 (indicating no reviews)
asheville_df['review_scores_rating'] = asheville_df['review_scores_rating'].fillna(0)

# Create a dummy variable (has_review_scores) to indicate whether a listing has a review
asheville_df['has_review_scores'] = (asheville_df['review_scores_rating'] > 0).astype(int)

asheville_df.isnull().sum()

Replace missing values of review_scores_rating with 0. Then created a dummy variable has_review_scores to indicate whether a listing has a review. has_review_scores = 1 if review_scores_rating > 0, has_review_scores = 0 if review_scores_rating == 0 (no reviews). Taken directly from Assignment 1.

### 1.5 Fix Data Types and Encode Columns

In [ ]:
# Check data types
asheville_df.dtypes

In [ ]:
# Convert price to Numeric
asheville_df['price'] = asheville_df['price'].replace(r'[\$,]', '', regex=True).astype(float)

Convert price from string to numeric for analysis. Taken directly from Assignment 1.

In [ ]:
# Convert Boolean-like Columns
asheville_df['host_identity_verified'] = asheville_df['host_identity_verified'].map({'t': 1, 'f': 0})
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].map({'t': 1, 'f': 0})

Dummy encode categorical variables host_identity_verified and host_is_superhost. Taken directly from Assignment 1.

In [ ]:
# Encode Categorical Column

room_type_dummies = pd.get_dummies(asheville_df['room_type'], prefix='room_type', drop_first=True)
room_type_dummies = room_type_dummies.astype(int)

# Add new columns to the DataFrame
asheville_df = pd.concat([asheville_df, room_type_dummies], axis=1)
asheville_df.drop('room_type', axis=1, inplace=True)

asheville_df.head()

One-hot encode room_type due to multi-category. Taken directly from Assignment 1.

### 1.6 Handle Outliers

In [ ]:
# Visualize the original price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (Before Removing Top 1%)")
plt.xlabel("Price")
plt.show()

In [ ]:
# Remove top 1% of extreme price values
price_cap = asheville_df['price'].quantile(0.99)
asheville_df = asheville_df[asheville_df['price'] <= price_cap]

In [ ]:
# Visualize the cleaned price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (After Removing Top 1%)")
plt.xlabel("Price")
plt.show()

Handle outliers by removing the top 1% of extreme price values. Taken directly from Assignment 1.

### 1.7 Descriptive Statistics

In [ ]:
# Descriptive Statistics
asheville_df.describe()

---
## 2. Additional Features
New features added for the mini-project to enhance model performance. These are pulled from the original dataset (`listings_df`) using index alignment with our cleaned `asheville_df`.

In [ ]:
# Reference the full Asheville data from the original dataset (still in memory)
# Using index alignment so only rows that survived cleaning get new feature values
asheville_full = listings_df[listings_df['host_location'] == 'Asheville, NC']

In [ ]:
# Feature 1: amenities_count — count the number of amenities listed for each property
# The amenities column is stored as a string like '["Wifi", "Kitchen", "Free parking"]'
# We parse it by splitting on commas to get the count
asheville_df['amenities_count'] = asheville_full['amenities'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) else 0
)

# Feature 2: host_total_listings_count — total number of properties managed by the host
asheville_df['host_total_listings_count'] = asheville_full['host_total_listings_count']

# Feature 3: minimum_nights — minimum stay requirement for the listing
asheville_df['minimum_nights'] = asheville_full['minimum_nights']

In [ ]:
# Check for any missing values in the new features
print(asheville_df[['amenities_count', 'host_total_listings_count', 'minimum_nights']].isnull().sum())
print()

# Impute any missing values with median (safe default for skewed numeric data)
for col in ['amenities_count', 'host_total_listings_count', 'minimum_nights']:
    asheville_df[col] = asheville_df[col].fillna(asheville_df[col].median())

In [ ]:
# Quick summary of the new features
asheville_df[['amenities_count', 'host_total_listings_count', 'minimum_nights']].describe()

In [ ]:
# Preview the updated dataframe with new features
asheville_df.head()

### Why These Features Were Chosen

Three additional features were selected to enhance the model beyond the Assignment 1 baseline:

**1. amenities_count** — The number of amenities a listing offers (e.g., Wifi, kitchen, free parking, pool, etc.). Properties with more amenities generally provide a higher-quality guest experience and can justify charging a premium. This feature captures the overall "value offering" of a property in a single number, and I expect it to have a meaningful positive correlation with price since well-equipped listings tend to command higher nightly rates.

**2. host_total_listings_count** — The total number of properties managed by the host across the Airbnb platform. This distinguishes professional hosts (who manage many properties) from individual hosts (who may only have one). Professional hosts often operate more like businesses with standardized pricing strategies, and they may price differently than casual hosts. This feature could reveal whether host scale influences pricing, which is a valuable business insight for both hosts and the platform.

**3. minimum_nights** — The minimum stay requirement set by the host. This is a strong indicator of the host's pricing strategy: listings with very short minimum stays (1–2 nights) are typically oriented toward tourists and short-term visitors, and tend to have higher per-night prices. Listings with longer minimum stays (e.g., 7+ nights or 30+ nights) often offer discounted nightly rates to attract longer-term guests. This feature captures pricing strategy behavior that the other features do not.

All three features are fully populated in the raw dataset (no missing values), which avoids introducing additional imputation noise. Together, they add information about property quality (amenities), host behavior (listings count), and booking strategy (minimum nights) — three distinct dimensions that the original Assignment 1 features did not capture.

---
## 3. Modeling Preparation
*Feature selection, train/test split, normalization, and helper functions.*

*(To be added)*

---
## 4. Model A: Baseline Neural Network (Assignment 1 Model)
*Input → Dense(64, ReLU) → Dense(1)*

*(To be added)*

---
## 5. Model B: Deeper Neural Network
*Input → Dense(128, ReLU) → Dense(64, ReLU) → Dense(1)*

*(To be added)*

---
## 6. Model C: Enhanced Features + Early Stopping
*Includes new features + Input → Dense(128, ReLU) → Dense(64, ReLU) → Dense(1) + EarlyStopping*

*(To be added)*

---
## 7. Model Comparison
*Side-by-side metrics (MAE, MSE, R²) and learning curves for all three models.*

*(To be added)*

---
## 8. Discussion and Business Insights
*Which model performs best? Did new features improve predictions? Other insights.*

*(To be added)*

---
## 9. GenAI Usage and References

*(To be added)*